In [1]:
import ollama

print("Ollama imported successfully.")


Ollama imported successfully.


In [2]:
response = ollama.chat(
    model="llama3.2:3b",
    messages=[
        {
            "role": "user",
            "content": "Reply with exactly: Python connected to SupportIQ LLM"
        }
    ]
)

print(
    response["message"]["content"]
)

Python connected to SupportIQ LLM


Import Libraries


In [3]:
import pandas as pd
import numpy as np
import pyarrow as pa

print("Pandas:", pd.__version__)
print("PyArrow:", pa.__version__)

Pandas: 2.3.3
PyArrow: 25.0.1


In [4]:
import os

rag_path = "../data/rag/supportiq_embeddings.parquet"

print("Exists:", os.path.exists(rag_path))
print("Size:", os.path.getsize(rag_path), "bytes")

Exists: True
Size: 22954918 bytes


In [5]:
with open(rag_path, "rb") as file:
    first_bytes = file.read(20)

print(first_bytes)

b'PAR1\x15\x00\x15\x90\xb4\x10\x15\xa8\xec\x06\x15\xf8\xf1\xd7\x94\x0e'


In [6]:
rag_path = "../data/rag/supportiq_embeddings.parquet"

with open(rag_path, "rb") as file:
    file.seek(-4, 2)
    last_bytes = file.read(4)

print("Last 4 bytes:", last_bytes)

Last 4 bytes: b'PAR1'


Read the exported data

In [7]:
rag_path = "../data/rag/supportiq_embeddings.parquet"

rag_df = pd.read_parquet(
    rag_path
)

print(
    "RAG records loaded:",
    len(rag_df)
)

RAG records loaded: 9600


Check columns

In [8]:
print(
    rag_df.columns
)

Index(['ticket_id', 'category', 'sub_category', 'product', 'priority',
       'issue_description', 'resolution', 'knowledge_text', 'embedding'],
      dtype='object')


Inspect a few records

In [9]:
rag_df[
    [
        "ticket_id",
        "category",
        "issue_description",
        "resolution"
    ]
].head()

,ticket_id,category,issue_description,resolution
0,TKT0924820,Time Management,Employee cannot access the benefit enrollment ...,Employee was provided with the required instru...
1,TKT0924977,Benefits,Manager cannot complete the performance review,System behavior was verified and corrected
2,TKT0925555,Performance,Employee is unable to request leave,System behavior was verified and corrected
3,TKT0926646,Benefits,Employee cannot access the application,Issue was resolved after configuration update
4,TKT0930365,Payroll,Employee is unable to update their profile,User permissions were corrected


Verify the embeddings survived the export

In [10]:
first_embedding = rag_df[
    "embedding"
].iloc[0]

print(
    type(first_embedding)
)

print(
    "Embedding dimension:",
    len(first_embedding)
)

<class 'numpy.ndarray'>
Embedding dimension: 384


Create the embedding matrix

In [11]:
embedding_matrix = np.array(
    rag_df["embedding"].tolist()
)

print(
    "Embedding matrix shape:",
    embedding_matrix.shape
)

Embedding matrix shape: (9600, 384)


Local Vector Search

Import Sentence Transformer

In [13]:
from sentence_transformers import SentenceTransformer

print("Sentence Transformers imported successfully.")

c:\Users\91840\OneDrive\Documents\PYTHON PROJECTS\support-iq\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Sentence Transformers imported successfully.


Load the same embedding model

In [14]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully.")

c:\Users\91840\OneDrive\Documents\PYTHON PROJECTS\support-iq\.venv\lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\91840\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\91840\OneDrive\Documents\PYTHON PROJECTS\support-iq\.v

Embedding model loaded successfully.


Test a new support question

In [15]:
question = "My salary statement is not showing"

query_embedding = embedding_model.encode(
    question
)

print(
    "Query embedding dimension:",
    len(query_embedding)
)

Query embedding dimension: 384


Import cosine similarity

In [16]:
from sklearn.metrics.pairwise import cosine_similarity

Calculate similarity

In [17]:
similarity_scores = cosine_similarity(
    [query_embedding],
    embedding_matrix
)[0]

print(
    "Number of similarity scores:",
    len(similarity_scores)
)

print(
    "Highest similarity:",
    similarity_scores.max()
)

Number of similarity scores: 9600
Highest similarity: 0.5413588470212527


Attach the scores

In [18]:
results = rag_df.copy()

results["similarity_score"] = similarity_scores

In [19]:
results = results.sort_values(
    by="similarity_score",
    ascending=False
)

Get Top 3

In [20]:
top_results = results.head(3)

top_results[
    [
        "category",
        "sub_category",
        "issue_description",
        "resolution",
        "similarity_score"
    ]
]

,category,sub_category,issue_description,resolution,similarity_score
904,Payroll,Overtime,Employee is unable to view their pay statement,Data was corrected in the system,0.541359
4273,Payroll,Overtime,Employee is unable to view their pay statement,Data was corrected in the system,0.539171
4205,Recruitment,Offer,Employee is unable to view their pay statement,Data was corrected in the system,0.538288


Turn it into a reusable function

In [21]:
def search_similar_tickets(question, top_k=3):

    query_embedding = embedding_model.encode(
        question
    )

    similarity_scores = cosine_similarity(
        [query_embedding],
        embedding_matrix
    )[0]

    results = rag_df.copy()

    results["similarity_score"] = similarity_scores

    results = results.sort_values(
        by="similarity_score",
        ascending=False
    )

    return results.head(top_k)

Test the function

In [22]:
question = "My salary statement is not showing"

similar_tickets = search_similar_tickets(
    question
)

similar_tickets[
    [
        "category",
        "sub_category",
        "issue_description",
        "resolution",
        "similarity_score"
    ]
]

,category,sub_category,issue_description,resolution,similarity_score
904,Payroll,Overtime,Employee is unable to view their pay statement,Data was corrected in the system,0.541359
4273,Payroll,Overtime,Employee is unable to view their pay statement,Data was corrected in the system,0.539171
4205,Recruitment,Offer,Employee is unable to view their pay statement,Data was corrected in the system,0.538288


Build the RAG Context

In [23]:
def build_context(similar_tickets):

    context = ""

    for index, row in similar_tickets.iterrows():

        context = context + (
            f"\nHistorical Case:\n"
            f"Category: {row['category']}\n"
            f"Sub Category: {row['sub_category']}\n"
            f"Issue: {row['issue_description']}\n"
            f"Resolution: {row['resolution']}\n"
        )

    return context

In [24]:
context = build_context(
    similar_tickets
)

print(context)


Historical Case:
Category: Payroll
Sub Category: Overtime
Issue: Employee is unable to view their pay statement
Resolution: Data was corrected in the system

Historical Case:
Category: Payroll
Sub Category: Overtime
Issue: Employee is unable to view their pay statement
Resolution: Data was corrected in the system

Historical Case:
Category: Recruitment
Sub Category: Offer
Issue: Employee is unable to view their pay statement
Resolution: Data was corrected in the system



Build the RAG Prompt

In [25]:
def build_rag_prompt(question, context):

    prompt = f"""
You are SupportIQ, an AI customer support assistant.

Answer the user's support question using the historical
support cases provided below.

Do not invent unsupported resolutions.

If the historical cases do not contain enough information,
state that further investigation may be required.

User Question:
{question}

Historical Support Cases:
{context}

Provide the answer in this format:

1. Likely Issue
2. Recommended Resolution
3. Escalation Recommendation
"""

    return prompt

Create the prompt

In [26]:
question = "My salary statement is not showing"

similar_tickets = search_similar_tickets(
    question
)

context = build_context(
    similar_tickets
)

rag_prompt = build_rag_prompt(
    question,
    context
)

print(rag_prompt)


You are SupportIQ, an AI customer support assistant.

Answer the user's support question using the historical
support cases provided below.

Do not invent unsupported resolutions.

If the historical cases do not contain enough information,
state that further investigation may be required.

User Question:
My salary statement is not showing

Historical Support Cases:

Historical Case:
Category: Payroll
Sub Category: Overtime
Issue: Employee is unable to view their pay statement
Resolution: Data was corrected in the system

Historical Case:
Category: Payroll
Sub Category: Overtime
Issue: Employee is unable to view their pay statement
Resolution: Data was corrected in the system

Historical Case:
Category: Recruitment
Sub Category: Offer
Issue: Employee is unable to view their pay statement
Resolution: Data was corrected in the system


Provide the answer in this format:

1. Likely Issue
2. Recommended Resolution
3. Escalation Recommendation



Send the RAG Prompt to Ollama 🚀

In [27]:
response = ollama.chat(
    model="llama3.2:3b",
    messages=[
        {
            "role": "user",
            "content": rag_prompt
        }
    ]
)

answer = response["message"]["content"]

print(answer)

1. Likely Issue
The employee may be unable to view their pay statement due to incorrect or outdated data in the system.

2. Recommended Resolution
Further investigation is required to identify the root cause of the issue. The employee's pay statement should be reviewed manually to determine the correct data and ensure it is accurately reflected in the system.

3. Escalation Recommendation
This issue should be escalated to a supervisor or payroll specialist to investigate and resolve the issue, and to verify the accuracy of the employee's pay statement.


Combine Everything Into One Function

In [28]:
def supportiq_rag(question):

    # Step 1: Search similar historical tickets
    similar_tickets = search_similar_tickets(
        question,
        top_k=3
    )

    # Step 2: Build context
    context = build_context(
        similar_tickets
    )

    # Step 3: Build RAG prompt
    rag_prompt = build_rag_prompt(
        question,
        context
    )

    # Step 4: Send prompt to local LLM
    response = ollama.chat(
        model="llama3.2:3b",
        messages=[
            {
                "role": "user",
                "content": rag_prompt
            }
        ]
    )

    # Step 5: Extract answer
    answer = response["message"]["content"]

    return answer

Test Your SupportIQ Assistant

In [29]:
question = "My salary statement is not showing"

answer = supportiq_rag(
    question
)

print(answer)

1. Likely Issue
The issue with the employee's salary statement not showing may be related to the payroll data being corrected in the system, which could be an indication of an internal process or data correction issue.

2. Recommended Resolution
Further investigation is required to determine the cause of the issue. Data has been corrected in the system in previous cases, but in this instance, the issue persists. The IT department may need to verify the system's payroll data and ensure that the employee's information is accurate and up-to-date.

3. Escalation Recommendation
Due to the lack of clear resolution, the issue should be escalated to a higher management level or the payroll manager for further investigation and resolution.


Add a Relevance Threshold

First inspect similarity scores

In [30]:
test_questions = [
    "My salary statement is not showing",
    "I cannot enroll in health insurance",
    "I cannot submit my time entry",
    "My manager cannot complete my performance review",
    "How do I cook pasta?",
    "What is the capital of France?"
]

for question in test_questions:

    results = search_similar_tickets(
        question,
        top_k=3
    )

    highest_score = results[
        "similarity_score"
    ].iloc[0]

    print(
        question,
        "->",
        round(highest_score, 3)
    )

My salary statement is not showing -> 0.541
I cannot enroll in health insurance -> 0.521
I cannot submit my time entry -> 0.665
My manager cannot complete my performance review -> 0.685
How do I cook pasta? -> 0.048
What is the capital of France? -> 0.023


Set the threshold

In [31]:
SIMILARITY_THRESHOLD = 0.50

Replace your old supportiq_rag() function

In [32]:
def supportiq_rag(question):

    # Search for similar historical tickets
    similar_tickets = search_similar_tickets(
        question,
        top_k=3
    )

    # Get the highest similarity score
    highest_score = similar_tickets[
        "similarity_score"
    ].iloc[0]

    # Check if the question is relevant
    if highest_score < SIMILARITY_THRESHOLD:

        return (
            "I could not find sufficiently relevant historical "
            "support cases for this question. "
            "Further investigation may be required."
        )

    # Build context from similar tickets
    context = build_context(
        similar_tickets
    )

    # Build the RAG prompt
    rag_prompt = build_rag_prompt(
        question,
        context
    )

    # Send the prompt to Llama
    response = ollama.chat(
        model="llama3.2:3b",
        messages=[
            {
                "role": "user",
                "content": rag_prompt
            }
        ]
    )

    # Get the answer
    answer = response["message"]["content"]

    return answer

Test a support question

In [33]:
question = "My salary statement is not showing"

answer = supportiq_rag(
    question
)

print(answer)

1. Likely Issue
The likely issue is that the employee's pay statement is not being generated or displayed due to an error or issue with the system.

2. Recommended Resolution
Further investigation may be required to determine the root cause of the issue, as the historical cases provided only mention data being corrected in the system as a resolution. I recommend checking the system logs and payroll records to identify the exact cause of the issue.

3. Escalation Recommendation
Escalate this issue to the Payroll Department for further investigation and assistance.


Test an unrelated question

In [34]:
question = "How do I cook pasta?"

answer = supportiq_rag(
    question
)

print(answer)

I could not find sufficiently relevant historical support cases for this question. Further investigation may be required.


Improve RAG Grounding

Replace build_rag_prompt()

In [35]:
def build_rag_prompt(question, context):

    prompt = f"""
You are SupportIQ, an AI customer support assistant.

Your job is to answer the user's question using ONLY the
historical support cases provided below.

IMPORTANT RULES:

1. Do not invent resolutions, teams, managers, procedures,
   or escalation paths that are not mentioned in the
   historical cases.

2. Base the recommended resolution only on the provided
   historical resolutions.

3. If multiple historical resolutions are available,
   summarize the most relevant ones.

4. Do not claim that a resolution will definitely fix
   the issue.

5. If the historical cases do not provide enough information,
   clearly state that further investigation is required.

6. Recommend escalation only when the historical cases
   support escalation or when the available information
   is insufficient.

User Question:
{question}

Historical Support Cases:
{context}

Provide the response in exactly this structure:

1. Likely Issue:
Explain the likely issue based on the historical cases.

2. Recommended Resolution:
Provide a resolution supported by the historical cases.

3. Escalation Recommendation:
State whether further investigation or escalation may be required.
"""

    return prompt

Test it again

In [36]:
question = "My salary statement is not showing"

answer = supportiq_rag(
    question
)

print(answer)

1. Likely Issue:
Based on the historical cases, the likely issue is that the employee's pay statement is not showing due to data discrepancies in the system.

2. Recommended Resolution:
According to the historical cases, a likely resolution is that the data was corrected in the system, but since the same issue is reported twice, further verification is needed to confirm this resolution.

3. Escalation Recommendation:
Further investigation is recommended to confirm the resolution and ensure that the issue has been fully resolved.


Test another support problem

In [37]:
question = "I cannot enroll in health insurance"

answer = supportiq_rag(
    question
)

print(answer)

1. Likely Issue:
The likely issue is that the employee is unable to access the health insurance benefit enrollment page, which has been an ongoing issue in the historical cases.

2. Recommended Resolution:
The recommended resolution is to check the configuration update mentioned in the historical cases, as it may have caused the issue.

3. Escalation Recommendation:
Further investigation is required as the historical cases only mention a configuration update as a resolution, but do not provide any additional information on how to resolve the issue.


Test another domain

In [38]:
question = "My manager cannot complete my performance review"

answer = supportiq_rag(
    question
)

print(answer)

1. Likely Issue:
The likely issue is that there is an issue with user permissions or access to perform the performance review.

2. Recommended Resolution:
The most relevant resolution from the historical cases is that a knowledge article was provided to address the issue. Therefore, I recommend checking the knowledge article for guidance on how to complete the performance review.

3. Escalation Recommendation:
Since a knowledge article was already provided and no other specific resolution is mentioned, further investigation may be required to identify the root cause of the issue.


Test the guardrail again

In [39]:
question = "How do I cook pasta?"

answer = supportiq_rag(
    question
)

print(answer)

I could not find sufficiently relevant historical support cases for this question. Further investigation may be required.
